In [ ]:
# "HuggingFaceTB/SmolVLM2-2.2B-Base"

# "HuggingFaceTB/SmolVLM2-2.2B-Base"

In [ ]:
# Chat Format
[
    {
        'role': 'system',
        'content': [{'type': 'text', 'text': 'You are a VLM specialized in charts'}]
    },
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': '<image_data>'},
            {'type': 'text', 'text': 'What is the highest value in this chart?'}
            ]
    },
    {
        'role': 'assistant',
        'content': [{'type': 'text', 'text': 'The highest value in this chart is 100.'}]
    }
]

In [ ]:
# Using vlm in a pipeline
from transformers import pipeline

# initialize pipeline with a VLM
pipe = pipeline('image-text-to-text', 'HuggingFaceTB/SmolVLM2-2.2B-Instruct', device_map="auto")

# Define conversation with an image
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg",},
            {'type': 'text', 'text': 'Describe the image.'}
        ]
    }
]

outputs = pipe(text=messages, max_new_tokens=100, return_full_text=False)

# Generate response with pipeline
response = pipe(messages, max_new_tokens=128, temperature=0.7)

print(response[0]['generated_text'][-1]['content'])

config.json:   0%|          | 0.00/3.64k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/63.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

/home/monster/Desktop/codes/ML-Workbook/myvenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:773: UserWarning: Not enough free disk space to download the file. The expected file size is: 4027.83 MB. The target location /home/monster/.cache/huggingface/hub/models--HuggingFaceTB--SmolVLM2-2.2B-Instruct/blobs only has 3260.95 MB free disk space.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

/home/monster/Desktop/codes/ML-Workbook/myvenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:773: UserWarning: Not enough free disk space to download the file. The expected file size is: 4027.83 MB. The target location /home/monster/.cache/huggingface/hub/models--HuggingFaceTB--SmolVLM2-2.2B-Instruct/blobs only has 3260.16 MB free disk space.
  warnings.warn(


In [ ]:
# !pip install num2words

In [1]:
# VLM Via Transformers
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from transformers.image_utils import load_image

device = 'cuda' if torch.cuda.is_available() else 'cpu'

quant_config = BitsAndBytesConfig(load_in_4bit=True)
model_name = 'HuggingFaceTB/SmolVLM2-2.2B-Instruct'
model = AutoModelForImageTextToText.from_pretrained(model_name, quantization_config=quant_config, device_map=device)
processor = AutoProcessor.from_pretrained(model_name)

# Load image
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"
image = load_image(image_url)

# create input messages
messages = [
    {
        'role': 'user',
        'content': [
            {
                'type': 'image',
                'image': image
            },
            {
                'type': 'text',
                'text': "Can you describe the image?"
            }
        ]
    }, # can add other messages
]

# Prepare inputs
inputs = processor.apply_chat_template(
    messages, 
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt").to(device)

# inputs = processor(prompt, images=image, return_tensors="pt").to(device)
# inputs = processor(prompt, return_tensors="pt").to(device)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=500)
generated_texts = processor.batch_decode(
    generated_ids, 
    skip_special_tokens=True)[0]

# Extract only assistant response
assistant_response = generated_texts.split('Assistant: ')[1].strip()

print(assistant_response)

Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

The image is a beautiful close-up of a bee on a pink flower. The bee is in the center of the flower, surrounded by the petals. The flower is vibrant and colorful, with shades of pink and yellow. The bee is small and black, with a fuzzy body. The background is a blur of green leaves, suggesting that the flower is in a garden or a natural setting. The bee is in the center of the flower, surrounded by the petals. The flower is vibrant and colorful, with shades of pink and yellow. The bee is small and black, with a fuzzy body. The background is a blur of green leaves, suggesting that the flower is in a garden or a natural setting.


In [ ]:
# SFT with VLM
from trl import SFTTrainer

training_args = SFTTrainer(
    output_dir='./sft_model',
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=5e-5,
    save_steps=1000,
    bf16=True,
    gradient_checkpointing=True,
    gradient_accumulation_steps=16,
    logging_steps=50
)